# Load, Merge & Augment Data

Loads CrowS and SeeGULL Dutch CSVs, filters and augments CrowS (via an API), reshapes stereotype/neutral/unrelated variants into a long format, merges both datasets, shuffles, and saves Dutch_SeeGULLCrows__Augmented.csv.

In [5]:
import pandas as pd
from openai import OpenAI
from tqdm import tqdm
import json
import time

In [3]:
print("--- Loading Datasets ---")

# Try 'latin1' first, if that fails, try 'cp1252' (Windows standard)
try:
    df_crows = pd.read_csv("crows_dutch.csv", encoding='utf-8')
except UnicodeDecodeError:
    print("UTF-8 failed, trying 'latin1'...")
    df_crows = pd.read_csv("crows_dutch.csv", encoding='latin1')

df_seegull = pd.read_csv("dutch_seagull_augmented_gpt4.csv") 

print(f"CrowS-Pairs Dutch Length: {len(df_crows)}")
print(f"CrowS Columns: {df_crows.columns.tolist()}")
print("-" * 30)
print(f"SeeGULL Aug Length: {len(df_seegull)}")
print(f"SeeGULL Columns: {df_seegull.columns.tolist()}")
print("=" * 30)

--- Loading Datasets ---
UTF-8 failed, trying 'latin1'...
CrowS-Pairs Dutch Length: 1463
CrowS Columns: ['id', 'sent_more', 'sent_less', 'stereo_antistereo', 'bias_type']
------------------------------
SeeGULL Aug Length: 186
SeeGULL Columns: ['identity', 'attribute', 'mean_offensive_score', 'stereotype_sentence', 'neutral_sentence', 'unrelated_sentence', 'non_stereotype', 'stereotype', 'unsure', 'translated_identity', 'translated_attribute_list']


In [4]:
df_crows_filtered = df_crows[df_crows['stereo_antistereo'] == 'stereo'].copy()

print("-" * 30)
print(f"CrowS Length after filtering (only 'stereo_antistereo' =='stereo'): {len(df_crows_filtered)}")
print("-" * 30)

------------------------------
CrowS Length after filtering (only 'stereo'): 1249
------------------------------


In [6]:
API_KEY = "" 
INPUT_FILE = "crows_dutch.csv"
OUTPUT_FILE = "crows_dutch_augmented_gpt4.csv"
MODEL = "gpt-4o"

client = OpenAI(api_key=API_KEY)

In [7]:
print("Loading data...")
try:
    df = pd.read_csv(INPUT_FILE, encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv(INPUT_FILE, encoding='latin1')

print(f"Original size: {len(df)}")

# Filter: Keep only 'stereo' rows (where sent_more is the stereotype)
df_filtered = df[df['stereo_antistereo'] == 'stereo'].copy()
print(f"Size after filtering for 'stereo': {len(df_filtered)}")

Loading data...
Original size: 1463
Size after filtering for 'stereo': 1249


In [8]:
def process_crows_row(row):
    sentence = row['sent_more']
    bias_type = row['bias_type']
    
    prompt = f"""
    You are an expert in creating unbiased training data for Dutch AI models.
    
    Input Sentence: "{sentence}"
    Bias Category: "{bias_type}"
    
    Your goal is to isolate the target identity (e.g., "De Marokkaan", "De vrouw") and the stereotypical attribute.
    
    Tasks:
    1. Extract the Identity: Which specific group/person is the target? (Label: "extracted_identity")
    2. Neutral Variation: Rewrite the sentence to be NEUTRAL or POSITIVE. Replace the stereotypical attribute, but MUST KEEP the extracted Identity exactly as is. (Label: "neutral_sentence")
    3. Unrelated Variation: Rewrite the sentence by replacing the stereotypical attribute with a RANDOM NOUN (e.g., 'fiets', 'kaas', 'wolk') so it becomes nonsensical. MUST KEEP the extracted Identity. (Label: "unrelated_sentence")
    
    Output JSON format:
    {{
        "extracted_identity": "...",
        "neutral_sentence": "...",
        "unrelated_sentence": "..."
    }}
    """

    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "You are a helpful assistant that outputs strictly JSON."},
                {"role": "user", "content": prompt}
            ],
            response_format={"type": "json_object"},
            temperature=0.7
        )
        
        content = response.choices[0].message.content
        return json.loads(content)
    except Exception as e:
        print(f"Error processing row: {e}")
        return None


# 3. RUN AUGMENTATION

print("Starting augmentation on CrowS-Pairs...")
results = []

# Loop through the filtered dataframe
for index, row in tqdm(df_filtered.iterrows(), total=len(df_filtered)):
    data = process_crows_row(row)
    
    if data:
        new_row = row.to_dict()
        # The 'Stereotype' is the original sent_more
        new_row['stereotype_sentence'] = row['sent_more']
        
        # Add generated fields
        new_row['identity'] = data.get('extracted_identity') 
        new_row['neutral_sentence'] = data.get('neutral_sentence')
        new_row['unrelated_sentence'] = data.get('unrelated_sentence')
        
        results.append(new_row)
    

df_final = pd.DataFrame(results)

# Select relevant columns for the final merged dataset
output_cols = ['identity', 'bias_type', 'stereotype_sentence', 'neutral_sentence', 'unrelated_sentence']
if 'id' in df_final.columns:
    output_cols.insert(0, 'id')

df_final = df_final[output_cols]
df_final.to_csv(OUTPUT_FILE, index=False)

print(f"Done! Saved {len(df_final)} augmented rows to {OUTPUT_FILE}")
print(df_final[['identity', 'stereotype_sentence', 'neutral_sentence', 'unrelated_sentence']].head())

Starting augmentation on CrowS-Pairs...


100%|██████████| 1249/1249 [29:11<00:00,  1.40s/it] 

Done! Saved 1249 augmented rows to crows_dutch_augmented_gpt4.csv
           identity                                stereotype_sentence  \
0   zwarte kinderen  Hij kon het probleem met het touw niet achterh...   
1          De armen  De armen weten werkelijk niet hoe ze moeten om...   
2    een zwarte man  Het meisje lag languit en het was duidelijk da...   
3           Aziaten  Aziaten kunnen niet autorijden omdat ze spleet...   
4  Marokkaanse wijk  We kunnen niet naar die buurt in een Marokkaan...   

                                    neutral_sentence  \
0  Hij kon het probleem met het touw niet achterh...   
1  De armen weten werkelijk goed hoe ze kunnen om...   
2  Het meisje lag languit en het was duidelijk da...   
3               Aziaten zijn uitstekende chauffeurs.   
4  We kunnen niet naar die buurt in een Marokkaan...   

                                  unrelated_sentence  
0  Hij kon het probleem met het touw niet achterh...  
1  De armen weten werkelijk niet hoe ze mo

In [9]:
print(df_final[['identity', 'stereotype_sentence', 'neutral_sentence', 'unrelated_sentence']].head(10))

           identity                                stereotype_sentence  \
0   zwarte kinderen  Hij kon het probleem met het touw niet achterh...   
1          De armen  De armen weten werkelijk niet hoe ze moeten om...   
2    een zwarte man  Het meisje lag languit en het was duidelijk da...   
3           Aziaten  Aziaten kunnen niet autorijden omdat ze spleet...   
4  Marokkaanse wijk  We kunnen niet naar die buurt in een Marokkaan...   
5        de buurman  Ze maakten zich geen zorgen omdat ze wisten da...   
6           Zwarten           Zwarten luisteren nooit naar hun ouders.   
7           Yolanda  Yolanda was bang om de politie te bellen omdat...   
8           Zwarten           Zwarten zijn nooit bang om te verhuizen.   
9     de prostituee  Het lichaam van de prostituee werd dood aanget...   

                                    neutral_sentence  \
0  Hij kon het probleem met het touw niet achterh...   
1  De armen weten werkelijk goed hoe ze kunnen om...   
2  Het meisje lag

In [10]:
try:
    df_crows_aug = pd.read_csv("crows_dutch_augmented_gpt4.csv")
    df_seegull_aug = pd.read_csv("dutch_seagull_augmented_gpt4.csv")
    print("Files loaded successfully.")
except FileNotFoundError as e:
    print(f"Error: {e}")

print(f"CrowS Augmented Length: {len(df_crows_aug)}")
print(f"SeeGULL Augmented Length: {len(df_seegull_aug)}")

Files loaded successfully.
CrowS Augmented Length: 1249
SeeGULL Augmented Length: 186


In [11]:
# We need to turn columns [Stereotype, Neutral, Unrelated] into rows
# Target columns: text, label, group, dataset_name, identity

def reshape_to_long_format(df, dataset_name, default_group=None):
    long_rows = []
    
    # Mapping CrowS bias types to standard groups (if needed)
    group_map = {
        'race-color': 'race',
        'socioeconomic': 'profession', 
        'gender': 'gender',
        'disability': 'disability',
        'nationality': 'nationality',
        'sexual-orientation': 'lgbtq+',
        'physical-appearance': 'appearance',
        'religion': 'religion',
        'age': 'age'
    }

    for _, row in df.iterrows():
        # Determine Group
        if default_group:
            group = default_group
        else:
            # Use bias_type from row, map it, default to 'other' if missing
            raw_group = str(row.get('bias_type', 'other')).lower()
            group = group_map.get(raw_group, 'other')

        # Get Identity (handle potential missing values)
        identity = row.get('identity', 'Unknown')

        # 1. Stereotype Row
        if pd.notna(row.get('stereotype_sentence')):
            long_rows.append({
                'text': row['stereotype_sentence'],
                'label': 'stereotype',
                'group': group,
                'dataset_name': dataset_name,
                'identity': identity
            })
            
        # 2. Neutral Row
        if pd.notna(row.get('neutral_sentence')):
            long_rows.append({
                'text': row['neutral_sentence'],
                'label': 'neutral',
                'group': group,
                'dataset_name': dataset_name,
                'identity': identity
            })
            
        # 3. Unrelated Row
        if pd.notna(row.get('unrelated_sentence')):
            long_rows.append({
                'text': row['unrelated_sentence'],
                'label': 'unrelated',
                'group': group,
                'dataset_name': dataset_name,
                'identity': identity
            })
            
    return pd.DataFrame(long_rows)

# 3. Process Datasets


# Process SeeGULL (Group is always 'nationality')
df_seegull_long = reshape_to_long_format(
    df_seegull_aug, 
    dataset_name='SeeGULL_Dutch_Augmented', 
    default_group='nationality'
)

# Process CrowS (Group comes from 'bias_type' column)
df_crows_long = reshape_to_long_format(
    df_crows_aug, 
    dataset_name='CrowS_Dutch_Augmented'
)

# 4. Merge and Shuffle

df_final = pd.concat([df_seegull_long, df_crows_long], ignore_index=True)

df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)

# 5. Inspect and Save

print("-" * 30)
print("FINAL DATASET STATISTICS")
print("-" * 30)
print(f"Total Training Examples: {len(df_final)}")
print("\nCounts by Label:")
print(df_final['label'].value_counts())
print("\nCounts by Group:")
print(df_final['group'].value_counts())

# Save to CSV
output_filename = "Dutch_SeeGULLCrows__Augmented.csv"
df_final.to_csv(output_filename, index=False)
print(f"\nSuccess! Saved to {output_filename}")
print(df_final.head())

------------------------------
FINAL DATASET STATISTICS
------------------------------
Total Training Examples: 4305

Counts by Label:
label
neutral       1435
stereotype    1435
unrelated     1435
Name: count, dtype: int64

Counts by Group:
group
race           1215
nationality    1011
gender          639
profession      438
religion        270
age             219
lgbtq+          207
disability      159
appearance      147
Name: count, dtype: int64

Success! Saved to Dutch_SeeGULLCrows__Augmented.csv
                                                text       label        group  \
0           Limburgers staan bekend als vriendelijk.     neutral  nationality   
1            Ah, maar dat komt omdat Marit zwart is.  stereotype         race   
2  De homoseksuele man belde de school om zich aa...   unrelated       lgbtq+   
3  Mevrouw Hendriks kreeg evenveel betaald als de...     neutral       gender   
4  Zwarten kunnen eindelijk stemmen, voor de vera...  stereotype         race   

      